# Baseline: Random Tote Sequencing

Random baseline for comparison. Generates many random tote-entry sequences and keeps the best one found.

In [1]:
import csv
import random
from pathlib import Path

# Choose which generated input run(s) to use.
# - RUN_ID = None  -> canonical inputs/
# - RUN_ID = int   -> one run folder inputs/runs/run_XXXX
# - RUN_ID = "all" -> all run folders under inputs/runs/
RUN_ID = "all"


def _resolve_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return paths[0]


def _get_input_bases(run_id):
    if run_id == "all":
        runs_root = _resolve_existing([Path("inputs/runs"), Path("../inputs/runs")])
        if not runs_root.exists():
            return []
        return sorted([p for p in runs_root.iterdir() if p.is_dir() and p.name.startswith("run_")])
    if run_id is None:
        return [_resolve_existing([Path("inputs"), Path("../inputs")])]
    run_name = f"run_{int(run_id):04d}"
    return [_resolve_existing([Path("inputs/runs") / run_name, Path("../inputs/runs") / run_name])]


output_base = _resolve_existing([Path("outputs"), Path("../outputs")])
output_base.mkdir(parents=True, exist_ok=True)
OUT = output_base

NUM_CONVEYORS = 4
PLACE_TIME = 1.75
TOTE_SWITCH_TIME = 4.0
BIN_SWITCH_TIME = 0.75
N_RANDOM_TRIALS = 2000
random.seed(42)

# Composite objective = total_time - OPTIONALITY_LAMBDA * optionality_score
OPTIONALITY_LAMBDA = 0.35
OPT_EDGE_WEIGHT = 1.0
OPT_BRANCH_WEIGHT = 0.6
OPT_RARE_WEIGHT = 0.4


def _coerce_int(v):
    s = v.strip()
    if s == "":
        return None
    try:
        return int(float(s))
    except ValueError:
        return None


def _read_rows(p):
    with p.open("r", newline="") as f:
        return list(csv.reader(f))


def build_blocks():
    ir = _read_rows(INPUT_ITEMTYPES)
    qr = _read_rows(INPUT_QUANTITIES)
    tr = _read_rows(INPUT_TOTES)
    n = max(len(ir), len(qr), len(tr))

    tote_bins = {}
    tote_items = {}
    for i in range(n):
        row_i = ir[i] if i < len(ir) else []
        row_q = qr[i] if i < len(qr) else []
        row_t = tr[i] if i < len(tr) else []
        w = max(len(row_i), len(row_q), len(row_t))
        for j in range(w):
            item = _coerce_int(row_i[j]) if j < len(row_i) else None
            qty = _coerce_int(row_q[j]) if j < len(row_q) else None
            tote = _coerce_int(row_t[j]) if j < len(row_t) else None
            if item is None or qty is None or tote is None or qty <= 0:
                continue
            tote_bins.setdefault(tote, []).extend([i + 1] * qty)
            tote_items.setdefault(tote, []).extend([item] * qty)

    blocks = {}
    for tote, bins in tote_bins.items():
        b = sorted(bins)
        blocks[tote] = {
            "first_bin": b[0],
            "last_bin": b[-1],
            "units": len(b),
            "internal_switches": sum(1 for k in range(1, len(b)) if b[k] != b[k - 1]),
            "items": tote_items[tote],
        }
    return blocks


def transition_cost(prev_tote, prev_last_bin, tote, blocks):
    if prev_tote is None:
        return 0.0
    c = TOTE_SWITCH_TIME
    if prev_last_bin != blocks[tote]["first_bin"]:
        c += BIN_SWITCH_TIME
    return c


def block_cost(tote, blocks):
    b = blocks[tote]
    return b["units"] * PLACE_TIME + b["internal_switches"] * BIN_SWITCH_TIME


def build_optionality_terms(blocks):
    totes = sorted(blocks.keys())
    first_bins = {t: blocks[t]["first_bin"] for t in totes}
    last_bins = {t: blocks[t]["last_bin"] for t in totes}

    compat = set()
    branch = {t: 0 for t in totes}
    for i in totes:
        for j in totes:
            if i == j:
                continue
            if last_bins[i] == first_bins[j]:
                compat.add((i, j))
                branch[i] += 1

    bin_freq = {}
    for t in totes:
        b = first_bins[t]
        bin_freq[b] = bin_freq.get(b, 0) + 1

    node_coeff = {}
    for t in totes:
        rarity = 1.0 / bin_freq[first_bins[t]]
        node_coeff[t] = OPT_BRANCH_WEIGHT * branch[t] - OPT_RARE_WEIGHT * rarity

    return {"compat": compat, "node_coeff": node_coeff}


def sequence_metrics(seq, blocks, terms):
    n = len(seq)
    total_time = 0.0
    optionality = 0.0

    prev_tote = None
    prev_last_bin = None
    for pos, tote in enumerate(seq, start=1):
        total_time += transition_cost(prev_tote, prev_last_bin, tote, blocks) + block_cost(tote, blocks)
        optionality += terms["node_coeff"][tote] * (n + 1 - pos)
        if prev_tote is not None and (prev_tote, tote) in terms["compat"]:
            optionality += OPT_EDGE_WEIGHT
        prev_tote = tote
        prev_last_bin = blocks[tote]["last_bin"]

    objective = total_time - OPTIONALITY_LAMBDA * optionality
    return total_time, optionality, objective


def write_sorter(seq, blocks, out_path):
    cols = {0: "circle", 1: "pentagon", 2: "trapezoid", 3: "triangle", 4: "star", 5: "moon", 6: "heart", 7: "cross"}
    rows = {}
    for tote in seq:
        # Conveyors are 1..4 for lab submission format
        conv = ((blocks[tote]["first_bin"] - 1) % NUM_CONVEYORS) + 1
        rows.setdefault(conv, {name: 0 for name in cols.values()})
        for shape in blocks[tote]["items"]:
            if shape in cols:
                rows[conv][cols[shape]] += 1

    out = []
    for conv in sorted(rows):
        r = {"conv_num": conv}
        r.update(rows[conv])
        out.append(r)

    with out_path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["conv_num", "circle", "pentagon", "trapezoid", "triangle", "star", "moon", "heart", "cross"])
        w.writeheader()
        for r in out:
            w.writerow(r)


def write_item_offload(seq, blocks, out_path):
    rows = []
    pos = 0
    for tote in seq:
        for item_type in blocks[tote]["items"]:
            pos += 1
            rows.append({"sequence_pos": pos, "item_type": item_type})

    with out_path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["sequence_pos", "item_type"])
        w.writeheader()
        w.writerows(rows)


all_bases = _get_input_bases(RUN_ID)
if not all_bases:
    raise RuntimeError("No input runs found. Ensure inputs/runs exists or set RUN_ID appropriately.")

aggregate = []
for base in all_bases:
    INPUT_ITEMTYPES = base / "order_itemtypes.csv"
    INPUT_QUANTITIES = base / "order_quantities.csv"
    INPUT_TOTES = base / "orders_totes.csv"

    blocks = build_blocks()
    terms = build_optionality_terms(blocks)
    totes = sorted(blocks.keys())

    best_seq = None
    best_objective = None
    best_total_time = None
    best_optionality = None
    for _ in range(N_RANDOM_TRIALS):
        seq = totes[:]
        random.shuffle(seq)
        total_time, optionality_score, objective_score = sequence_metrics(seq, blocks, terms)
        if best_objective is None or objective_score < best_objective:
            best_objective = objective_score
            best_total_time = total_time
            best_optionality = optionality_score
            best_seq = seq[:]

    if RUN_ID == "all":
        run_out = OUT / "baseline_random_runs" / base.name
    elif RUN_ID is None:
        run_out = OUT
    else:
        run_out = OUT / "baseline_random_runs" / base.name
    run_out.mkdir(parents=True, exist_ok=True)

    with (run_out / "baseline_random_tote_sequence.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["sequence_pos", "tote"])
        w.writeheader()
        for i, tote in enumerate(best_seq, start=1):
            w.writerow({"sequence_pos": i, "tote": tote})

    write_sorter(best_seq, blocks, run_out / "optimized_input_from_baseline_random.csv")
    write_item_offload(best_seq, blocks, run_out / "baseline_random_tote_item_plan.csv")

    row = {
        "run_name": base.name,
        "trials": N_RANDOM_TRIALS,
        "total_time": best_total_time,
        "optionality_score": best_optionality,
        "objective_score": best_objective,
        "n_totes": len(best_seq),
        "n_units": sum(b["units"] for b in blocks.values()),
    }
    with (run_out / "baseline_random_summary.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        w.writeheader()
        w.writerow(row)
    aggregate.append(row)

if RUN_ID == "all":
    with (OUT / "baseline_random_all_runs_summary.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(aggregate[0].keys()))
        w.writeheader()
        w.writerows(aggregate)
    print(f"Processed {len(aggregate)} runs.")
    print("Wrote aggregate: outputs/baseline_random_all_runs_summary.csv")
else:
    print(f"Random baseline best objective over {N_RANDOM_TRIALS} trials: {aggregate[0]['objective_score']:.3f}")
    print(f"Random baseline total time: {aggregate[0]['total_time']:.3f}")
    print("Wrote run outputs under outputs/")

Processed 500 runs.
Wrote aggregate: outputs/baseline_random_all_runs_summary.csv
